# Notebook 06 — Metacognitive Bias (Supplementary Tables 6–8)

Tests whether metacognitive measures are sensitive to confidence *bias* — a systematic tendency to use high or low confidence ratings independently of accuracy.

**Method (Xue et al., 2021)**:
1. **Recode 1** (high-confidence bias): subtract 1 from all confidence ratings; if any trial hits the minimum, bump it up by 1. This shifts all ratings toward *higher* confidence.
2. **Recode 2** (low-confidence bias): replace the maximum rating with max − 1. This shifts ratings toward *lower* confidence.
3. Compute all 20 measures under each recoding.
4. Test recode2 − recode1 against zero with a one-sample t-test.

A measure with a significant effect is sensitive to confidence bias — a problem for comparing groups that differ in average confidence.

In [1]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


metasignal loaded successfully.


In [2]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


In [3]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


Statistical helper functions defined.


In [4]:
import matplotlib.pyplot as plt

## Xue et al. recoding function

In [5]:
def xue_recode(conf, rtype):
    """
    Recode confidence ratings per Xue et al. (2021).
    rtype=1: high-confidence bias (subtract 1, floor at min+1)
    rtype=2: low-confidence bias  (replace max with max-1)
    Returns new conf array with same length but one fewer category.
    """
    valid = conf[~np.isnan(conf)]
    if len(np.unique(valid)) < 3:
        return np.full_like(conf, np.nan, dtype=float)
    c = conf.copy().astype(float)
    if rtype == 1:
        c -= 1
        cmin = np.nanmin(c)
        c[c == cmin] = cmin + 1
    elif rtype == 2:
        cmax = np.nanmax(c)
        c[c == cmax] = cmax - 1
    return c

# Example: 4-rating scale
ex = np.array([1,2,3,4,4,3,2,1])
print("Original:  ", ex)
print("Recode 1:  ", xue_recode(ex, 1).astype(int))   # high-conf bias
print("Recode 2:  ", xue_recode(ex, 2).astype(int))   # low-conf bias


Original:   [1 2 3 4 4 3 2 1]
Recode 1:   [1 1 2 3 3 2 1 1]
Recode 2:   [1 2 3 3 3 3 2 1]


## Load precomputed bias data

In [6]:
ha_bias       = np.load(os.path.join(OUT, 'haddara_mle.npz'))['bias']      # (70,2,20)
ma_bias       = np.load(os.path.join(OUT, 'maniscalco_mle.npz'))['bias']   # (22,2,20)
sh_bias_full  = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['bias']      # (20,3,2,20)
sh_bias       = np.nanmean(sh_bias_full, axis=1)                            # average contrasts → (20,2,20)
print("Haddara bias:", ha_bias.shape)
print("Maniscalco bias:", ma_bias.shape)
print("Shekhar bias (avg over contrasts):", sh_bias.shape)


Haddara bias: (70, 2, 20)
Maniscalco bias: (22, 2, 20)
Shekhar bias (avg over contrasts): (20, 2, 20)


## Supplementary Tables 6–8

In [7]:
cohens_d_lbl = "Cohen's d"
EXCL = {"d'", "Criterion"}   # unaffected by conf recoding (response-only measures)

REPORTED = {
    6: {"meta-d'":2.584,"AUC2":0.688,"Gamma":-4.331,"Phi":1.257,
        "DeltaConf":1.034,"M-Ratio":1.994,"Gamma-Diff":2.334,"Confidence":24.538},
    7: {"meta-d'":2.711,"AUC2":3.794,"Phi":5.262,"DeltaConf":5.242,"Confidence":17.328},
    8: {"AUC2":2.804,"Gamma":-4.284,"Phi":5.133,"DeltaConf-Ratio":2.992,"Confidence":13.845},
}

for tnum, bias_arr, label in [
    (6, ha_bias,  "Table 6: Haddara (n=70)"),
    (7, ma_bias,  "Table 7: Maniscalco (n=22)"),
    (8, sh_bias,  "Table 8: Shekhar (n=20)"),
]:
    delta = bias_arr[:, 1, :] - bias_arr[:, 0, :]   # recode2 - recode1
    rep   = REPORTED[tnum]
    print(f"\nSupplementary {label}")
    print("=" * 72)
    print(f"  {'Measure':<20} {'t':>7} {'p':>8} {'sig':>4} {cohens_d_lbl:>10} {'t (MATLAB)':>12}")
    print("  " + "-"*65)
    for m, name in enumerate(MEASURE_NAMES):
        if name in EXCL: continue
        t, df, p, d, lo, hi = ttest_1samp(delta[:, m])
        rep_t = rep.get(name, float('nan'))
        match = "" if np.isnan(t) else ("✓" if abs(t-rep_t)<abs(rep_t)*0.05+0.1 else "~") if not np.isnan(rep_t) else ""
        t_s = f"{t:7.3f}" if not np.isnan(t) else "    NaN"
        d_s = f"{d:10.3f}" if not np.isnan(d) else "       NaN"
        p_s = f"{p:8.4f}" if not np.isnan(p) else "     nan"
        rt_s = f"{rep_t:12.3f}" if not np.isnan(rep_t) else "           —"
        print(f"  {name:<20} {t_s} {p_s} {p_stars(p):>4} {d_s} {rt_s}  {match}")



Supplementary Table 6: Haddara (n=70)
  Measure                    t        p  sig  Cohen's d   t (MATLAB)
  -----------------------------------------------------------------
  meta-d'                2.318   0.0234    *      0.277        2.584  ~
  AUC2                   0.688   0.4936   ns      0.082        0.688  ✓
  Gamma                 -4.331   0.0000  ***     -0.518       -4.331  ✓
  Phi                    1.257   0.2129   ns      0.150        1.257  ✓
  DeltaConf              1.034   0.3047   ns      0.124        1.034  ✓
  M-Ratio                1.795   0.0770   ns      0.215        1.994  ✓
  AUC2-Ratio             1.176   0.2436   ns      0.141            —  
  Gamma-Ratio            0.510   0.6117   ns      0.061            —  
  Phi-Ratio              1.062   0.2920   ns      0.127            —  
  DeltaConf-Ratio        1.664   0.1006   ns      0.199            —  
  M-Diff                 2.316   0.0236    *      0.277            —  
  AUC2-Diff              1.237   0.22

## Visualisation — effect of Xue recoding

In [8]:
FIGS = os.path.join(REPO, 'notebooks', 'figures')
os.makedirs(FIGS, exist_ok=True)
COLORS = {'Haddara':'#d55e00','Maniscalco':'#0072b2','Shekhar':'#009e73'}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (bias_arr, label, color) in zip(axes, [
    (ha_bias,  'Haddara (n=70)',    COLORS['Haddara']),
    (ma_bias,  'Maniscalco (n=22)', COLORS['Maniscalco']),
    (sh_bias,  'Shekhar (n=20)',    COLORS['Shekhar']),
]):
    delta = bias_arr[:, 1, :] - bias_arr[:, 0, :]
    means = np.nanmean(delta, axis=0)
    sems  = np.nanstd(delta, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(delta), axis=0))
    x = np.arange(N_MEAS)
    ax.bar(x, means, yerr=sems, color=color, alpha=0.8, capsize=3)
    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_xticks(x); ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=7)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Recode2 − Recode1 ± SEM')
plt.suptitle('Supplementary Figure 3: Metacognitive Bias (Xue Recoding)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'fig_xue_recode.png'), dpi=150, bbox_inches='tight')
print("Saved fig_xue_recode.png")
plt.show()


Saved fig_xue_recode.png
